# 🧬 ESMFold Structure Prediction Example

This notebook demonstrates how to predict protein 3D structures using **ESMFold**.

## 📖 What is ESMFold?

ESMFold is a fast protein structure prediction model from Meta AI that predicts 3D structures directly from amino acid sequences using a language model approach, without requiring multiple sequence alignments (MSAs).

### Key Features:
- ⚡ **Fast predictions** - 60x faster than AlphaFold2
- 🧬 **MSA-free** - No evolutionary information needed
- 💯 **Confidence scores** - pLDDT and pTM metrics

## Input/Output Schema

### `ESMFoldInput`
| Field | Type | Description |
|-------|------|-------------|
| complexes | List[StructurePredictionComplex] | List of complexes to predict structures for. Each complex can contain one or more protein chains. Total length across all chains must not exceed 2,400 residues. |

**Supported entity types:** `protein`
**Allows chain modifications:** No

### `ESMFoldConfig`
| Field | Type | Default | Description |
|-------|------|---------|-------------|
| residue_idx_offset | int | `512` | Residue numbering gap between chains in multi-chain structures |
| chain_linker | str | `"G" * 25` | Sequence to link chains (default: 25 glycines) |
| max_batch_residues | int | `1200` | Maximum total residues per inference batch (to prevent GPU memory overflow) |
| device | str | `"cuda"` | Device to run the model on (e.g., `"cuda"`, `"cpu"`) |
| verbose | bool | `False` | Whether to print status messages during execution |
| keep_on_gpu | bool | `False` | Whether to keep model on device after inference |

### `ESMFoldOutput` (alias for `StructurePredictionOutput`)
| Field | Type | Description |
|-------|------|-------------|
| structures | List[ProteinStructure] | List of predicted structures, one per input complex. Each structure contains 3D coordinates and confidence metrics (avg_plddt, ptm, avg_pae). pLDDT is normalized to 0.0-1.0 scale. |
| metadata | dict | Additional information about the prediction run |

## Imports

In [1]:
from bio_programming_tools.tools.structure_prediction.esmfold.esmfold import (
    run_esmfold,
    ESMFoldInput,
    ESMFoldConfig,
)
from pathlib import Path

## 🔍 Single Protein Structure Prediction (GFP Example)

Let's predict the 3D structure of GFP (Green Fluorescent Protein).

### Configuration options:
- 🖥️ **`device`** - GPU ("cuda") or CPU
- 📊 **`verbose`** - Show progress messages

In [2]:
# GFP sequence
gfp_sequence = "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTFSYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITHGMDELYK"

# Create input
inputs = ESMFoldInput(complexes=[gfp_sequence])

# Configure ESMFold
config = ESMFoldConfig(
    verbose=False,
    device="cuda",  # Change to "cpu" if no GPU available
)

# Run prediction
result = run_esmfold(inputs, config)
gfp_structure = result.structures[0]

# Print metrics
print(f"  Length: {gfp_structure.num_residues} residues")
print(f"  Average pLDDT: {gfp_structure.avg_plddt:.3f}")
print(f"  pTM score: {gfp_structure.ptm:.3f}")

Folding structures (ESMFold): 100%|██████████| 1/1 [00:12<00:00, 12.26s/structure]

  Length: 238 residues
  Average pLDDT: 0.393
  pTM score: 0.445


### 🎨 Visualize GFP Structure

In [3]:
gfp_structure.visualize()

## 🎯 Batch Predictions

Predict multiple protein structures in a single run for better efficiency:

In [4]:
# Multiple protein sequences to predict
sequences = [
    "MYKCELCGKAFSNQSSLSRHQRTHTGEKPFQC",  # Zinc finger
    "MKFLKFSLLTAVLLSVVFAFSSCGDDDDTISN",  # Signal peptide example
    "GSHMKQLEDKVEELLSKNYHLENEVARLKKLV",  # Small helical protein
    "MKFLKFSLLTAVLLSVVFAFSSCGDDDDTISN"   # Signal peptide example
]

# Create batch input
batch_input = ESMFoldInput(complexes=sequences)

# Run batch prediction
batch_config = ESMFoldConfig(verbose=False, device="cuda")
batch_result = run_esmfold(batch_input, batch_config)

# Display results for all structures
print(f"\n✅ Predicted {len(batch_result.structures)} structures!\n")
for i, structure in enumerate(batch_result.structures):
    print(f"Structure {i+1}:")
    print(f"  Length: {structure.num_residues} residues")
    print(f"  Average pLDDT: {structure.avg_plddt:.3f}")
    print(f"  pTM score: {structure.ptm:.3f}")
    print()

Folding structures (ESMFold): 100%|██████████| 4/4 [00:11<00:00,  2.76s/structure]


✅ Predicted 4 structures!

Structure 1:
  Length: 32 residues
  Average pLDDT: 0.752
  pTM score: 0.594

Structure 2:
  Length: 32 residues
  Average pLDDT: 0.652
  pTM score: 0.594

Structure 3:
  Length: 32 residues
  Average pLDDT: 0.873
  pTM score: 0.594

Structure 4:
  Length: 32 residues
  Average pLDDT: 0.652
  pTM score: 0.594



## 💾 Export Results

Save predicted structures for further analysis in other tools like PyMOL, ChimeraX, or VMD.

### Supported formats:
- 📄 **PDB** - Standard protein data bank format, widely compatible
- 📊 **mmCIF** - Modern crystallographic information file, supports larger structures

The B-factor column contains the normalized pLDDT scores (0.0-1.0) for confidence visualization.

In [ ]:
# Create output directory
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export results to pdb files
result.export(name="gfp_structure", export_path=output_dir, file_format="pdb")

# Export the batch results to a directory
batch_result.export(name="batch_structures", export_path=output_dir, file_format="pdb")